# Mirror approvals

Copies approvals to the eventhouse so Activator can see them, and
remediations back so the report can show status.

This notebook is **generated**. Edit
`validation/build_mirror_notebook.py` and regenerate.

## Why this exists

Activator cannot watch a SQL database. Its sources are Power BI
semantic models, KQL querysets, Real-Time Dashboards, Eventstreams,
Fabric events and Azure events. So the approval is written where the
user data function can reach it with a managed connection, and copied
to where the rule can see it.

## Why a notebook and not a pipeline

A pipeline was the first attempt and its definition cannot be validated
outside the tenant. This is generated, drift tested, and its cells are
executed by unit tests.

**Python, not Spark.** It moves a handful of rows, and a Spark session
takes longer to start than this takes to run. On a one minute schedule
that is the difference between working and not.

## Run it every minute

That interval is the approval latency: about a minute here, plus about
a minute for the Activator rule to poll.

## 1. Parameters

In [ ]:
# Filled in at deployment. Empty in source control, like every other
# deployment value in this repo.
WORKSPACE_ID = ""
SQL_DATABASE = "SQLDB_AgentEval"
KUSTO_URI = ""
KUSTO_DB = "EH_AgentEval"


## 2. Connect to both stores

In [ ]:
import json
import urllib.request
from datetime import datetime, timezone

import notebookutils

# The SQL side. connect_to_artifact is a Python notebook API and is the reason
# this is not a Spark notebook.
sql = notebookutils.data.connect_to_artifact(SQL_DATABASE, WORKSPACE_ID, "SQLDatabase")

kusto_token = notebookutils.credentials.getToken(KUSTO_URI)


def kusto(csl, endpoint="query"):
    request = urllib.request.Request(
        f"{KUSTO_URI}/v1/rest/{endpoint}",
        data=json.dumps({"db": KUSTO_DB, "csl": csl}).encode("utf-8"),
        method="POST",
        headers={"Authorization": f"Bearer {kusto_token}",
                 "Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=120) as response:
        return json.loads(response.read().decode("utf-8"))


def kusto_rows(result):
    table = result["Tables"][0]
    names = [c["ColumnName"] for c in table["Columns"]]
    return [dict(zip(names, row)) for row in table["Rows"]]


def sql_rows(result):
    """Rows from a SQL query, as a list of dicts.

    connect_to_artifact returns None rather than an empty frame when a SELECT
    matches nothing, so `list(result)` raises TypeError on the ordinary case
    of there being nothing to mirror.

    That case is the steady state, not an edge case. This notebook runs every
    minute and almost every run has no new approvals, so a version that only
    worked when there was something to copy failed on every scheduled run and
    succeeded whenever anybody tested it by hand right after approving
    something. It looked like a scheduling problem for a day.
    """
    if result is None:
        return []
    if hasattr(result, "to_dict"):
        return result.to_dict("records")
    return list(result)


def escape(value):
    """Escape a value for a Kusto double quoted string literal.

    Backslash first. Escaping the quotes before the backslashes turns `a\\`
    into `a\\"` and ends the literal early, which is both a broken command
    and the shape of an injection.
    """
    if value is None:
        return ""
    return str(value).replace("\\", "\\\\").replace('"', '\\"')


## 3. Approvals to the eventhouse

Unmirrored rows only, marked as mirrored after the copy succeeds.

In [ ]:
# Approvals that have not reached the eventhouse yet.
pending = sql.query("""
    SELECT approval_id, approved_ts, question_id, instruction_target,
           proposed_instruction, decision, approved_by, approver_oid,
           source, note
    FROM dbo.approvals
    WHERE mirrored_ts IS NULL
    ORDER BY approved_ts
""")

rows = sql_rows(pending)
print(f"{len(rows)} approval(s) to mirror")

mirrored = []
for row in rows:
    stamp = row["approved_ts"]
    if not isinstance(stamp, str):
        stamp = stamp.strftime("%Y-%m-%dT%H:%M:%S.%fZ")

    # One command per row rather than a batch. A batch that fails halfway
    # would leave some rows copied and none marked, and the retry would copy
    # them again.
    kusto(
        ".set-or-append eval_approvals <| print "
        f'approval_id="{escape(row["approval_id"])}", '
        f"approved_ts=datetime({stamp}), "
        f'question_id="{escape(row["question_id"])}", '
        f'instruction_target="{escape(row["instruction_target"])}", '
        f'proposed_instruction="{escape(row["proposed_instruction"])}", '
        f'decision="{escape(row["decision"])}", '
        f'approved_by="{escape(row["approved_by"])}", '
        f'note="{escape(row["note"])}"',
        endpoint="mgmt",
    )
    mirrored.append(row["approval_id"])
    print(f"  mirrored {row['question_id']} ({row['decision']})")

# Marked only after the copy succeeds. Marking first would lose an approval on
# any failure, and a lost approval is a change a person authorised that never
# happened and that nothing reports.
for approval_id in mirrored:
    sql.query(
        "UPDATE dbo.approvals SET mirrored_ts = SYSUTCDATETIME() "
        f"WHERE approval_id = '{approval_id}'"
    )

print(f"marked {len(mirrored)} as mirrored")


## 4. Remediations back to SQL

So `dbo.open_approvals` closes and the report shows applied and
verified states.

In [ ]:
# The return leg. Without it dbo.open_approvals never sees a remediation
# land, every applied approval looks open forever, and the function refuses
# every second approval for a question.
applied = kusto_rows(kusto("""
    eval_remediations
    | summarize arg_max(recorded_ts, *) by remediation_id
    | project remediation_id, recorded_ts, applied_ts, approval_id, question_id,
              instruction_target, instruction, approved_by, applied_by,
              dry_run, persisted, verified, verified_ts, verified_run_id
"""))

print(f"{len(applied)} remediation(s) in the eventhouse")


def sql_literal(value):
    if value is None or value == "":
        return "NULL"
    if isinstance(value, bool):
        return "1" if value else "0"
    return "N'" + str(value).replace("'", "''") + "'"


written = 0
for row in applied:
    # Upsert on the key, because the eventhouse is append only and a
    # remediation gains a corrected row when it becomes verified. An insert
    # would give the report two rows for one remediation, one permanently
    # stale.
    values = ", ".join(sql_literal(row[c]) for c in (
        "remediation_id", "recorded_ts", "applied_ts", "approval_id",
        "question_id", "instruction_target", "instruction", "approved_by",
        "applied_by", "dry_run", "persisted", "verified", "verified_ts",
        "verified_run_id",
    ))
    sql.query(f"""
        MERGE dbo.remediations AS t
        USING (SELECT {sql_literal(row['remediation_id'])} AS remediation_id) AS s
        ON t.remediation_id = s.remediation_id
        WHEN MATCHED THEN UPDATE SET
            recorded_ts = {sql_literal(row['recorded_ts'])},
            verified    = {sql_literal(row['verified'])},
            verified_ts = {sql_literal(row['verified_ts'])},
            verified_run_id = {sql_literal(row['verified_run_id'])}
        WHEN NOT MATCHED BY TARGET THEN
            INSERT (remediation_id, recorded_ts, applied_ts, approval_id,
                    question_id, instruction_target, instruction, approved_by,
                    applied_by, dry_run, persisted, verified, verified_ts,
                    verified_run_id)
            VALUES ({values});
    """)
    written += 1

print(f"upserted {written} remediation(s) into SQL")


## 5. What healthy looks like

In [ ]:
# What a person should see if this is healthy. An approval with an old
# approved_ts and a null mirrored_ts never reached the eventhouse, so the rule
# never fired, and that is a query rather than a mystery.
print(sql.query("""
    SELECT
        (SELECT COUNT(*) FROM dbo.approvals)                       AS approvals,
        (SELECT COUNT(*) FROM dbo.approvals WHERE mirrored_ts IS NULL) AS unmirrored,
        (SELECT COUNT(*) FROM dbo.open_approvals)                  AS open_approvals,
        (SELECT COUNT(*) FROM dbo.remediations)                    AS remediations
"""))
